In [ ]:
import re

import polars as pl

import plotly.io as pio
import plotly.express as px

# import textstat
from wordfreq import zipf_frequency

In [ ]:
# Configure data, models, device, and plot configurations
DATA_PATH = "../../data/train.csv"
ZIPF_PLOT = "../../plots/inverted_zipf_frequency.png"

OPTION_COLS = ["A", "B", "C", "D", "E"]

COL1 = '#00040A'
COL2 = '#202124'
COL3 = '#E1E1E0'
GRID = '#525458'

TITLE_FONT_SIZE = 20
LABEL_FONT_SIZE = 16
TICK_FONT_SIZE = 12

PLOT_WDTH = 1150
PLOT_HGHT = 800
pio.renderers.default = 'notebook'

data = pl.read_csv(DATA_PATH)

data = data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

data = data.with_columns(
    (
        pl.lit("Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_options")
)

ids = data["id"].to_list()
prompts = data["prompt"].to_list()

In [ ]:
# Define function to get inverted Zipf frequency [get_text_rarity]
def get_text_rarity(text: str) -> float:
    if not text or not isinstance(text, str):
        return 0.0
    
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    
    if not words:
        return 0.0
    
    rarity_scores = [8.0 - zipf_frequency(word, 'en') for word in words]
    
    return sum(rarity_scores) / len(rarity_scores)

In [ ]:
# Get inverted Zipf frequency for prompt, mcq_options, and mcq_query
data = data.with_columns([
    pl.col("prompt").map_elements(get_text_rarity, return_dtype=pl.Float64).alias("prompt_rarity"),
    pl.col("mcq_options").map_elements(get_text_rarity, return_dtype=pl.Float64).alias("option_rarity"),
    pl.col("mcq_query").map_elements(get_text_rarity, return_dtype=pl.Float64).alias("query_rarity")
])

In [ ]:
# Construct plotting dataframe for inverted Zipf frequency data
plot_zipf_data = pl.DataFrame({
    "X": data["prompt_rarity"],
    "Y": data["option_rarity"],
    "Total Text Complexity": data["query_rarity"],
    "ID": ids,
    "Prompt": prompts
})

In [ ]:
# inverted_zipf_frequency
# Plot scatter plot using computed inverted Zipf frequency data
fig = px.scatter(
    plot_zipf_data, 
    x="X",
    y="Y",
    color="Total Text Complexity", 
    hover_data=["ID", "Prompt"],
    labels={
        "X": "Question Stem Complexity (Rarer Vocabulary)",
        "Y": "Options Complexity (Rarer Vocabulary)",
        "Total Text Complexity": "Total Text Complexity"
    },
    color_continuous_scale="Viridis"
)

fig.update_traces(marker=dict(size=4, opacity=0.8))

# Add the diagonal reference line
max_val = max(plot_zipf_data["X"].max(), plot_zipf_data["Y"].max()) + 1 # type: ignore
fig.add_shape(
    type="line", 
    line=dict(dash='dash', color="gray"),
    x0=0, 
    y0=0, 
    x1=max_val, 
    y1=max_val
)

fig.update_layout(
    width=PLOT_WDTH,
    height=PLOT_HGHT,
    paper_bgcolor=COL1,
    plot_bgcolor=COL1,
    font=dict(color=COL3),
    hovermode="closest",

    title=dict(
        text="MCQ Linguistic Complexity: Questions vs. Options (Zipf Rarity Index)",
        font=dict(color=COL3, size=TITLE_FONT_SIZE),
        x=0.5, xanchor="center"
    ),

    scene=dict(
        bgcolor=COL1,
        xaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        ),
        yaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        )
    )
)

fig.write_image(ZIPF_PLOT)

fig.show()